In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.GroupNorm(8, out_channels),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)

class ShipGeometryEncoder(nn.Module):

    def __init__(self, latent_dim=512):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32),       # 256 -> 128
            ConvBlock(32, 64),      # 128 -> 64
            ConvBlock(64, 128),     # 64 -> 32
            ConvBlock(128, 256),    # 32 -> 16
            ConvBlock(256, 512)     # 16 -> 8
        )
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(
            512 * 8 * 8,
            latent_dim
        )

    def forward(self, x):
        feature_map = self.features(x)
        z = self.fc(
            self.flatten(feature_map)
        )
        return z, feature_map


class RoomGraphEncoder(nn.Module):

    def __init__(self, input_dim=8, hidden_dim=64, output_dim=128):
        super().__init__()
        self.conv1 = SAGEConv(input_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, output_dim)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return x

In [8]:
import torch
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

# Node features:
# [area, mass, cabin, corridor, kitchen, lounge, engine, storage]
x = torch.tensor([
    [0.50, 0.17, 1, 0, 0, 0,  0,0],  # 0: Cabin 1
    [0.67, 0.25, 0, 1, 0, 0, 0, 0],  # 1: Corridor
    [0.47, 0.16, 1, 0, 0, 0, 0, 0],  # 2: Cabin 2
    [0.60, 0.38, 0, 0, 1, 0, 0, 0],  # 3: Kitchen
    [1.00, 0.42, 0, 0, 0, 1, 0, 0],  # 4: Lounge
    [0.83, 1.00, 0, 0, 0, 0, 1, 0],  # 5: Engine room
    [0.40, 0.21, 0, 0, 0, 0, 0, 1],  # 6: Storage
], dtype=torch.float)


# Define edges (bidirectional)
edge_index = torch.tensor([
    [0, 1, 1, 1, 1, 2, 3, 4, 4, 5, 5, 6],
    [1, 0, 2, 3, 4, 1, 1, 1, 5, 4, 6, 5]
], dtype=torch.long)

# Create PyG graph
graph = Data(
    x=x,
    edge_index=edge_index
)

model = RoomGraphEncoder(input_dim=8, hidden_dim=64, output_dim=128)
graph_features = model(graph.x, graph.edge_index)